# Chapter 18 — Don't Let the Optimizer Cheat

**Book alignment:** DSPy From First Principles, Chapter 18

**Question this notebook isolates:** Does the six-layer firewall block all nine constructed leakage attacks on one train case while passing its clean payload?


In [ ]:
from pathlib import Path
import importlib.util
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "dspy-from-first-principles" / "common").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/dspy-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "dspy-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

from common.data import canonical_split
from common.metrics import score_editorial_output

spec = importlib.util.spec_from_file_location(
    "ch16_leakage_run", str(EXP_ROOT / "ch16_leakage_attacks" / "run.py")
)
fw = importlib.util.module_from_spec(spec)
sys.modules["ch16_leakage_run"] = fw
spec.loader.exec_module(fw)

ALLOWED_REVISION = "teaching-revision-v1"
print("firewall layers:", ["forbidden_key", "exact_outcome_value", "source_lineage", "revision", "tool_capability", "holdout_overlap"])


## Schema checks names only

The same gold text renamed under an innocent key passes the forbidden-key audit. The content audit catches it, which is why a split plus a key list is not a firewall.


In [ ]:
split = canonical_split()
case = next(c for c in split.train if c.case_id == "ed-002")
clean = fw.clean_payload(case)
sneaky = {**clean, "retrieval_hint": case.reference_rewrite}
forbidden_values = {case.reference_rewrite.strip()}
schema_hit = fw.forbidden_key_audit(sneaky)
content_hit = fw.exact_outcome_value_audit(sneaky, forbidden_values=forbidden_values)
print("case:", case.case_id, "| schema detected:", schema_hit.detected, "| content detected:", content_hit.detected)
print("content reasons:", content_hit.reasons)


In [ ]:
assert schema_hit.detected is False
assert content_hit.detected is True
print("gold value under an innocent key defeats schema; content inspection is required")


## Nine attacks, one train case, zero admissions

The full constructed attack set runs against `ed-002` with a synthetic holdout probe ID, so the canonical holdout content is never read.


In [ ]:
probe = "synthetic-holdout-probe"
audit_ids = set(split.holdout_ids) | {probe}
results = []
for spec_attack in fw.attack_specs(case, allowed_revision=ALLOWED_REVISION):
    verdict = fw.run_firewall(
        payload=spec_attack["payload"],
        manifest=spec_attack["manifest"],
        forbidden_values=forbidden_values,
        target_case_id=case.case_id,
        target_source_id=case.source_id,
        target_source_group=case.source_group,
        forbidden_case_ids=audit_ids,
        holdout_ids=audit_ids,
        allowed_revision=ALLOWED_REVISION,
    )
    expected = set(spec_attack["expected_layers"])
    detected = set(verdict["detected_layers"])
    results.append((spec_attack["attack"], verdict["blocked"], expected <= detected, sorted(detected)))
clean_verdict = fw.run_firewall(
    payload=clean, manifest={}, forbidden_values=forbidden_values,
    target_case_id=case.case_id, target_source_id=case.source_id,
    target_source_group=case.source_group, forbidden_case_ids=audit_ids,
    holdout_ids=audit_ids, allowed_revision=ALLOWED_REVISION,
)
for name, blocked, contract, layers in results:
    print(f"{name:32s} blocked={blocked!s:5s} contract={contract!s:5s} layers={layers}")
print("clean payload blocked:", clean_verdict["blocked"], "| attacks:", len(results))


In [ ]:
assert len(results) == 9
assert all(blocked for _, blocked, _, _ in results)
assert all(contract for _, _, contract, _ in results)
assert clean_verdict["blocked"] is False
print("9/9 attacks blocked with expected layers; clean payload passes (no false positive)")


## The incentive to cheat is real

A contaminated candidate that copies the reference scores above the clean no-edit baseline. The inflated score measures the temptation; the firewall verdict measures whether the experiment would admit the evidence.


In [ ]:
clean_score = score_editorial_output(case, case.sentence)
oracle_score = score_editorial_output(case, case.reference_rewrite)
inflation = oracle_score.score - clean_score.score
print(f"clean no-edit score: {clean_score.score:.4f} | oracle reference score: {oracle_score.score:.4f} | inflation: {inflation:+.4f}")


In [ ]:
assert inflation > 0
print("forbidden evidence would have improved the apparent score: the firewall earns its keep")


## What we earned

A split is not a firewall and a forbidden-key list is not a firewall: gold information arrives under innocent keys, inside feedback, from duplicate sources, through future revisions, or via answer-bearing tools, and only the layered checks catch all of them.

Notebook 19 / Chapter 19 spends the sealed holdout once and builds the ordered gate that decides what an offline win may authorize.
